# Binary Predictor Pipeline

Запуск бинарного пайплайна: parse -> preprocess (3 ESP) -> majority vote классификация.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'tools').exists() and (project_root.parent / 'tools').exists():
    project_root = project_root.parent
if not (project_root / 'tools').exists() and (project_root.parent.parent / 'tools').exists():
    project_root = project_root.parent.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipelines.binary_predictor import BinaryPredictor

print(f'project_root: {project_root}')

project_root: /home/gleb/learning/CSI-activity-detection


In [2]:
artifacts_root = project_root / 'artifacts' / 'classic_ml_majority_vote_metrics_each_esp'

preproc_weights_dev1 = artifacts_root / 'dev1' / 'preprocessing.joblib'
preproc_weights_dev2 = artifacts_root / 'dev2' / 'preprocessing.joblib'
preproc_weights_dev3 = artifacts_root / 'dev3' / 'preprocessing.joblib'

clf_weights_dev1 = artifacts_root / 'dev1' / 'svm_model.joblib'
clf_weights_dev2 = artifacts_root / 'dev2' / 'svm_model.joblib'
clf_weights_dev3 = artifacts_root / 'dev3' / 'svm_model.joblib'

required_paths = [
    preproc_weights_dev1, preproc_weights_dev2, preproc_weights_dev3,
    clf_weights_dev1, clf_weights_dev2, clf_weights_dev3,
]
missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing artifacts:\n' + '\n'.join(str(p) for p in missing))

predictor = BinaryPredictor(
    preproc_weights_dev1=preproc_weights_dev1,
    preproc_weights_dev2=preproc_weights_dev2,
    preproc_weights_dev3=preproc_weights_dev3,
    clf_weights_dev1=clf_weights_dev1,
    clf_weights_dev2=clf_weights_dev2,
    clf_weights_dev3=clf_weights_dev3,
)

print('BinaryPredictor initialized')

BinaryPredictor initialized


In [3]:
# Пример запуска на одном test_* каталоге.
# Итог: 0 -> static / no motion, 1 -> dynamic / motion
import time

test_folder = project_root / 'wifi_data_set_fixed_1fps' / 'id_person_01' / 'label_00' / 'test_02_4'
start_time = time.time()
predicted_class = predictor.predict_from_test_folder(test_folder)
print(time.time() - start_time)
print(f'test_folder: {test_folder}')
print(f'predicted_class: {predicted_class}')

0.047373294830322266
test_folder: /home/gleb/learning/CSI-activity-detection/wifi_data_set_fixed_1fps/id_person_01/label_00/test_02_4
predicted_class: 1


In [ ]:
# Инференс на всем датасете используя пайплайн

In [ ]:
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
# import pandas as pd


# def true_class_from_label(label_name: str) -> int:
#     return 0 if label_name == 'label_00' else 1


# dataset_root = project_root / 'wifi_data_set_fixed_1fps'

# test_dirs = sorted(
#     p for p in dataset_root.rglob('test_*')
#     if p.is_dir() and any(p.glob('*.data'))
# )

# rows = []
# failed_rows = []

# for idx, test_dir in enumerate(test_dirs, start=1):
#     label_name = test_dir.parent.name
#     person_id = test_dir.parent.parent.name
#     test_id = test_dir.name

#     y_true = true_class_from_label(label_name)

#     try:
#         y_pred = predictor.predict_from_test_folder(test_dir)
#         rows.append({
#             'person_id': person_id,
#             'label': label_name,
#             'test_id': test_id,
#             'group_path': str(test_dir),
#             'y_true': y_true,
#             'y_pred': int(y_pred),
#         })
#     except Exception as exc:
#         failed_rows.append({
#             'person_id': person_id,
#             'label': label_name,
#             'test_id': test_id,
#             'group_path': str(test_dir),
#             'error': str(exc),
#         })

#     if idx % 200 == 0:
#         print(f'processed groups: {idx}/{len(test_dirs)}')

# results_df = pd.DataFrame(rows)
# failed_df = pd.DataFrame(failed_rows)

# print(f'Total groups found: {len(test_dirs)}')
# print(f'Successfully inferred: {len(results_df)}')
# print(f'Failed groups: {len(failed_df)}')

# if results_df.empty:
#     raise RuntimeError('No successful predictions; metrics cannot be computed')

# y_true = results_df['y_true'].to_numpy()
# y_pred = results_df['y_pred'].to_numpy()

# precision, recall, f1, support = precision_recall_fscore_support(
#     y_true,
#     y_pred,
#     labels=[0, 1],
#     zero_division=0,
# )

# metrics_df = pd.DataFrame({
#     'class': ['static / no motion', 'dynamic / motion'],
#     'precision': precision,
#     'recall': recall,
#     'f1_score': f1,
#     'support': support,
# })

# print(f"accuracy: {accuracy_score(y_true, y_pred):.4f}")
# print('Confusion matrix, rows=true, cols=pred:')
# print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
# print()
# print(classification_report(
#     y_true,
#     y_pred,
#     labels=[0, 1],
#     target_names=['static / no motion', 'dynamic / motion'],
#     zero_division=0,
# ))

# errors_df = results_df[results_df['y_true'] != results_df['y_pred']].copy()
# print(f'Total errors: {len(errors_df)}')

# metrics_df